# Multifactorial recoverability boundary + Frobenius-Schur analysis

Two CPU-only analyses that supply verified numbers for the paper. No GPU, no training, seconds to run.

1. **Second-invariant test (§ recoverability).** Irrep dimension predicts the recoverability boundary
   best. Does any *second* group invariant remove the residual it leaves? If none do, the boundary is
   genuinely multifactorial (the paper's stated claim). All invariants are computed exactly from group
   theory; the recoverability numbers are pasted from the 11-group experiment.
2. **Frobenius--Schur indicators.** Compute FS indicators for the relevant irreps and identify which
   groups carry a non-real (complex or quaternionic) gauge, where the real commutant, and thus the
   true gauge, departs from the naive $d^2$. This is the FS correction the Method section refers to.

Every printed number is produced on this run. Paste printed values into the paper, never remembered ones.

In [3]:
import numpy as np
np.set_printoptions(precision=4, suppress=True, linewidth=140)
OK = lambda b: "PASS" if b else "*** FAIL ***"
def spearman(x, y):
    x=np.asarray(x,float); y=np.asarray(y,float)
    rx=np.argsort(np.argsort(x)); ry=np.argsort(np.argsort(y))
    return float(np.corrcoef(rx, ry)[0, 1])
print("ready")

ready


## 1. Is the recoverability boundary multifactorial?

**Paste your measured `defect@0.2` per group** from the 11-group recoverability run. The values below
are from that experiment's summary table; replace them if you re-run so the paper and this analysis
use identical numbers.

In [4]:
# defect@0.2 per group  --  PASTE MEASURED VALUES from the 11-group recoverability experiment.
defect_02 = {
 "Z/6":0.078, "S3":0.462, "D4":0.378, "Q8":0.251, "D5":0.567, "Z/12":0.237,
 "D6":0.496, "A4":0.664, "S4":0.828, "A5":1.019, "S5":1.119,
}

# ---- exact group invariants (computed/known, not measured) ----
maxdim   = {"Z/6":1,"S3":2,"D4":2,"Q8":2,"D5":2,"Z/12":1,"D6":2,"A4":3,"S4":3,"A5":5,"S5":6}
order    = {"Z/6":6,"S3":6,"D4":8,"Q8":8,"D5":10,"Z/12":12,"D6":12,"A4":12,"S4":24,"A5":60,"S5":120}
nclass   = {"Z/6":6,"S3":3,"D4":5,"Q8":5,"D5":4,"Z/12":12,"D6":6,"A4":4,"S4":5,"A5":5,"S5":7}
exponent = {"Z/6":6,"S3":6,"D4":4,"Q8":4,"D5":10,"Z/12":12,"D6":6,"A4":6,"S4":12,"A5":30,"S5":60}
# total gauge = |G| - (# irreps);  # irreps = # conjugacy classes
gauge    = {g: order[g]-nclass[g] for g in order}

groups = list(defect_02)
y = np.array([defect_02[g] for g in groups])

print("Spearman rho of each invariant with defect@0.2 (all groups):")
invs = [("max irrep dim",maxdim),("group order",order),("total gauge |G|-#irr",gauge),
        ("num conj. classes",nclass),("exponent",exponent)]
for name, inv in invs:
    print(f"  {name:22s}: rho = {spearman([inv[g] for g in groups], y):+.3f}")

Spearman rho of each invariant with defect@0.2 (all groups):
  max irrep dim         : rho = +0.964
  group order           : rho = +0.855
  total gauge |G|-#irr  : rho = +0.964
  num conj. classes     : rho = -0.036
  exponent              : rho = +0.700


In [5]:
# Residual test: regress defect on max irrep dimension, then ask whether any SECOND
# invariant explains what dimension leaves. Small |rho| for all => multifactorial.
xd = np.array([maxdim[g] for g in groups], float)
A  = np.vstack([xd, np.ones_like(xd)]).T
beta, *_ = np.linalg.lstsq(A, y, rcond=None)
resid = y - A @ beta
print(f"After regressing out max irrep dim: residual std = {resid.std():.4f}\n")
print("Does a second invariant explain the residual?  (Spearman of invariant with residual)")
for name, inv in [("group order",order),("total gauge",gauge),
                  ("num classes",nclass),("exponent",exponent)]:
    r = spearman([inv[g] for g in groups], resid)
    verdict = "explains" if abs(r) > 0.6 else "does NOT explain"
    print(f"  {name:15s}: rho = {r:+.3f}   -> {verdict}")

# Within-dimension view: at fixed max-dim = 2, the defect still spreads widely.
sub = [g for g in groups if maxdim[g] == 2]
vals = {g: defect_02[g] for g in sub}
print(f"\nWithin fixed max irrep dim = 2  ({', '.join(sub)}):")
print(f"  defects: {vals}")
print(f"  spread (max - min) = {max(vals.values()) - min(vals.values()):.3f}")
print("\nCLAIM CHECK: if every second-invariant |rho| with the residual is well below 0.6,")
print("the boundary is multifactorial -- dimension predicts best but no single invariant closes it.")

After regressing out max irrep dim: residual std = 0.1106

Does a second invariant explain the residual?  (Spearman of invariant with residual)
  group order    : rho = +0.282   -> does NOT explain
  total gauge    : rho = +0.291   -> does NOT explain
  num classes    : rho = -0.273   -> does NOT explain
  exponent       : rho = +0.327   -> does NOT explain

Within fixed max irrep dim = 2  (S3, D4, Q8, D5, D6):
  defects: {'S3': 0.462, 'D4': 0.378, 'Q8': 0.251, 'D5': 0.567, 'D6': 0.496}
  spread (max - min) = 0.316

CLAIM CHECK: if every second-invariant |rho| with the residual is well below 0.6,
the boundary is multifactorial -- dimension predicts best but no single invariant closes it.


## 2. Frobenius--Schur indicators and non-real gauge

The FS indicator of an irrep is $\nu_\rho = \frac{1}{|G|}\sum_g \chi_\rho(g^2) \in \{+1, 0, -1\}$:
$+1$ real (orthogonal), $0$ complex (unitary), $-1$ quaternionic (symplectic). The real commutant per
copy is $1, 2, 4$ respectively, so gauge equals the naive $d^2$ only for real irreps. We compute $\nu$
directly from explicit irrep matrices, verifying each is a genuine representation first.

In [6]:
# ---- Q8: explicit 2D irrep from unit quaternions (quaternionic, expect nu = -1) ----
I2 = np.eye(2, dtype=complex)
# consistent unit-quaternion 2x2 realization:  i,j,k with i*j = k
qi = np.array([[1j,0],[0,-1j]])
qj = np.array([[0,1],[-1,0]])
qk = np.array([[0,1j],[1j,0]])
Q8 = {"1":I2, "-1":-I2, "i":qi, "-i":-qi, "j":qj, "-j":-qj, "k":qk, "-k":-qk}

def is_rep(mats):
    els = list(mats.values())
    ok = True
    for A in els:
        for B in els:
            if not any(np.allclose(A@B, C) for C in els): ok = False
    return ok
print("Q8 2D matrices form a group (closure):", OK(is_rep(Q8)))
print("Q8 relation i*j = k:", OK(np.allclose(qi@qj, qk)))
print("Q8 relation i^2 = -1:", OK(np.allclose(qi@qi, -I2)))

def fs_indicator(mats):
    els = list(mats.values()); n = len(els)
    # nu = mean over g of chi(g^2); need g^2 as a matrix (we have it: square each element)
    return float(np.real(np.mean([np.trace(M@M) for M in els])))

nu_Q8_2d = fs_indicator(Q8)
print(f"\nFS indicator, Q8 two-dimensional irrep:  nu = {nu_Q8_2d:+.3f}   "
      f"({'quaternionic' if nu_Q8_2d < -0.5 else 'real' if nu_Q8_2d > 0.5 else 'complex'})")

Q8 2D matrices form a group (closure): PASS
Q8 relation i*j = k: PASS
Q8 relation i^2 = -1: PASS

FS indicator, Q8 two-dimensional irrep:  nu = -1.000   (quaternionic)


In [7]:
# ---- Contrast: the 2D irreps of the symmetric/dihedral groups are REAL (nu = +1). ----
# S3 standard 2D irrep via its action on {(1,0),(0,1)} with the standard rep of S3 ~ D3.
def dihedral_2d(n):
    th = 2*np.pi/n
    r = np.array([[np.cos(th),-np.sin(th)],[np.sin(th),np.cos(th)]])
    s = np.array([[1,0],[0,-1]], float)
    els = {}
    R = np.eye(2)
    for a in range(n):
        els[f"r{a}"] = R.copy()
        els[f"sr{a}"] = s @ R
        R = r @ R
    return els

for nm, grp in [("S3 (=D3) 2D", dihedral_2d(3)), ("D4 2D", dihedral_2d(4)), ("D5 2D", dihedral_2d(5))]:
    els = list(grp.values())
    nu = float(np.real(np.mean([np.trace(M@M) for M in els])))
    print(f"FS indicator, {nm:12s}: nu = {nu:+.3f}  ({'real' if nu>0.5 else 'quaternionic' if nu<-0.5 else 'complex'})")

print("\nSUMMARY FOR THE PAPER:")
print("  Q8's only nonabelian gauge lives in a QUATERNIONIC (FS = -1) irrep; the symmetric and")
print("  dihedral groups' 2D irreps are REAL (FS = +1). Real commutant per copy: real=1, quaternionic=4.")
print("  So Q8's gauge is not the naive d^2 that fits the S_n family -- a concrete representation-")
print("  theoretic reason its recoverability sits off the pure dimension trend (defect 0.251, low for")
print("  |G|=8). Report the FS indicators; attribute Q8's off-trend position to its quaternionic gauge.")

FS indicator, S3 (=D3) 2D : nu = +1.000  (real)
FS indicator, D4 2D       : nu = +1.000  (real)
FS indicator, D5 2D       : nu = +1.000  (real)

SUMMARY FOR THE PAPER:
  Q8's only nonabelian gauge lives in a QUATERNIONIC (FS = -1) irrep; the symmetric and
  dihedral groups' 2D irreps are REAL (FS = +1). Real commutant per copy: real=1, quaternionic=4.
  So Q8's gauge is not the naive d^2 that fits the S_n family -- a concrete representation-
  theoretic reason its recoverability sits off the pure dimension trend (defect 0.251, low for
  |G|=8). Report the FS indicators; attribute Q8's off-trend position to its quaternionic gauge.


## What to copy into the paper

- **§ recoverability, multifactorial claim:** the per-invariant Spearman rho, the residual std after
  removing dimension, and the second-invariant-vs-residual rho values. The strong claim ("no second
  invariant removes the residual") is licensed only if every second-invariant |rho| is well below 0.6.
- **FS / Q8:** the FS indicators (Q8 2D = -1 quaternionic; dihedral 2D = +1 real), and the resulting
  statement that Q8's gauge departs from $d^2$. Include only if the residual analysis shows Q8 off-trend.

All numbers are produced on this run and validated by the closure/relation checks above.